# 3주차 · 영화 리뷰 감성 분석 (NLP) 🎬

이번 주 미니 대회는 **컴퓨터에게 글의 감정을 읽게 하기**! 네이버 영화 리뷰 텍스트를 보고 **긍정(1)인지 부정(0)인지** 맞히는 문제예요.

한 가지 함정이 있어요 — 긍정 리뷰가 전체의 약 25%뿐인 **불균형 데이터**입니다. 그래서 "다 부정!"이라고 찍으면 정확도는 75%처럼 보이지만, 평가 지표인 **F1 점수는 0점**이에요. F1은 소수 클래스(긍정)를 얼마나 잘 찾아내는지를 보는 지표거든요.

**진행 방법 (꼭 읽어주세요!)**
1. 셀을 **위에서부터 하나씩** `Shift+Enter`로 실행하세요.
2. **✏️ 표시된 셀만** 바꿔보면 됩니다.
3. 마지막 셀이 `submission.csv`를 다운로드해줘요 → 웹 **Submit 탭**에 업로드.

- 평가 지표: **F1** (높을수록 좋음, 0~1 사이)
- 기본 코드 그대로도 베이스라인은 이깁니다. 완주가 먼저! 😊

## 1. 데이터 불러오기  *(그냥 실행하세요)*

리뷰 데이터를 읽어옵니다. `document` 컬럼이 리뷰 텍스트, `target`이 정답(1=긍정, 0=부정)이에요. 텍스트가 비어있는(결측) 리뷰는 에러를 막기 위해 빈 문자열로 채웁니다.

실행하면 데이터 크기와 함께 `긍정 비율: 0.25` 정도가 찍혀요 — 이게 바로 아까 말한 불균형이에요!

In [ ]:
import pandas as pd  # 표 데이터를 다루는 라이브러리

# 데이터 공개 링크 (운영진이 채워둔 것 — 수정하지 마세요!)
TRAIN_URL = "https://raw.githubusercontent.com/nepersoned/dat-datasets/main/nlp/train.csv"
TEST_URL  = "https://raw.githubusercontent.com/nepersoned/dat-datasets/main/nlp/test.csv"

if TRAIN_URL:
    train = pd.read_csv(TRAIN_URL); test = pd.read_csv(TEST_URL)  # URL에서 바로 읽기
else:
    # (예비용) URL이 없을 때만 직접 업로드 — 보통 실행되지 않아요
    from google.colab import files; files.upload()
    train = pd.read_csv("train.csv"); test = pd.read_csv("test.csv")

# 텍스트가 비어있는(NaN) 리뷰가 있으면 에러가 나므로 빈 문자열("")로 채웁니다
train["document"] = train["document"].fillna("")
test["document"] = test["document"].fillna("")

print("train:", train.shape, "| test:", test.shape)
print("긍정 비율:", round(train["target"].mean(), 2))  # 0.25 근처 = 불균형!
train.head(3)  # 리뷰 3개 미리보기

## 2. ✏️ 여기를 바꿔보세요 — 텍스트를 숫자로 바꾸고 모델 학습

컴퓨터는 글자를 바로 이해하지 못해서, 먼저 **TF-IDF**라는 방법으로 텍스트를 숫자표로 바꿉니다. TF-IDF는 "이 리뷰에 어떤 단어가 얼마나 특징적으로 쓰였나"를 점수로 매기는 방법이에요. 그 숫자표를 **로지스틱 회귀**(가장 기본적인 분류 모델)에 넣어 긍정/부정을 배우게 합니다.

`class_weight='balanced'`가 핵심 — 수가 적은 긍정 리뷰를 틀리면 벌점을 더 크게 줘서, 불균형 데이터에서도 긍정을 놓치지 않게 해줘요.

실행하면 몇십 초 안에 끝나고, 마지막에 모델 정보가 출력되면 성공이에요.

**점수(F1) 올리는 실험 아이디어** — 하나씩 바꿔서 제출해보세요:
- `ngram_range=(1, 2)` → `(1, 3)`: 단어 1개·2개짜리 조합에 더해 3개짜리 표현("정말 재미 없다")까지 보기
- `max_features=50000` → `100000`: 기억하는 단어 수 늘리기
- `TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4))`: 단어 대신 **글자 조각** 단위로 보기 — 한국어처럼 띄어쓰기가 들쭉날쭉한 텍스트에 효과적일 때가 많아요
- 더 도전: `konlpy` 형태소 분석, 사전학습 모델(BERT 계열)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer  # 텍스트 → 숫자표 변환기
from sklearn.linear_model import LogisticRegression           # 기본 분류 모델

# ✏️ 여기를 바꿔보세요! ngram_range=(1,3), max_features=100000 등으로 실험
# ngram_range=(1, 2): 단어 1개짜리 + 연속된 2개짜리("재미 없다")를 모두 특징으로 사용
# max_features=50000: 자주 나온 단어/조합 5만 개까지만 기억
vec = TfidfVectorizer(ngram_range=(1, 2), max_features=50000)

Xtr = vec.fit_transform(train["document"])  # 학습 리뷰로 단어사전을 만들고 숫자표로 변환
Xte = vec.transform(test["document"])       # 같은 사전으로 test 리뷰도 변환

# class_weight="balanced": 수가 적은 긍정(25%)을 틀리면 벌점을 크게 → 불균형 대응 핵심!
# max_iter=1000: 학습 반복 횟수 상한 (수렴 경고 방지용)
clf = LogisticRegression(max_iter=1000, class_weight="balanced")
clf.fit(Xtr, train["target"])  # 학습 시작! (몇십 초 정도)

## 3. 예측 & 제출 파일 저장  *(실행하면 자동 다운로드)*

학습된 모델로 test 리뷰들의 긍정/부정을 예측하고 `submission.csv`로 저장합니다. 실행하면 파일이 **자동으로 다운로드**돼요.

👉 다운로드된 `submission.csv`를 웹사이트의 **Submit 탭**에 업로드하면 F1 점수가 나옵니다. 여러 번 제출할 수 있으니 위 셀을 바꿔가며 점수를 올려보세요!

In [ ]:
# 학습된 모델로 test 리뷰 각각의 긍정(1)/부정(0)을 예측
test["prediction"] = clf.predict(Xte)

# 제출 규칙: id, prediction 두 컬럼만 저장 (index=False: 행 번호 제외)
test[["id", "prediction"]].to_csv("submission.csv", index=False)
print("저장 완료")

try:
    # 코랩에서 실행 중이면 자동 다운로드
    from google.colab import files; files.download("submission.csv")
except Exception:
    print("왼쪽 파일탭에서 submission.csv를 내려받으세요.")

## 🆘 막히면 여기를 보세요

| 증상 | 해결 |
|---|---|
| `NameError: name 'train' is not defined` | 위쪽 셀을 건너뛰었어요. 맨 위부터 순서대로 실행하세요. |
| `ValueError: np.nan is an invalid document` | 1번 셀의 `fillna("")` 부분이 실행 안 된 거예요. 1번 셀부터 다시 실행. |
| 학습이 1분 넘게 걸림 | `max_features`를 크게 키우면 오래 걸려요. 정상이니 기다리거나 값을 줄이세요. |
| 데이터 로드 URL 에러 | 인터넷 연결 확인 후 셀을 한 번 더 실행해보세요. |

텍스트도 결국 숫자로 바꾸면 머신러닝이 된다는 것, 오늘의 핵심이었어요! 🙌